# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

```pseudocode
# Import warnings library and ignore all warnings.
```

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

```pseudocode
# Import environment variable loading utilities.
# Load environment variables from a .env file.
# Retrieve and set API keys for OpenAI and Hugging Face.
```

In [3]:
#!pip install pandas

```pseudocode
# Install the pandas library using pip.
```

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


```pseudocode
# Import google.colab drive utility.
# Mount Google Drive to access files.
```

In [5]:
import os
print(os.listdir("/content/drive/MyDrive/Ironhack_COLAB/lab-chains-in-langchain/lab-chains-in-langchain-main"))


['.env', 'README.md.gdoc', 'Data.csv', 'lab-chains-in-langchain.ipynb']


```pseudocode
# Import the os module.
# List the contents of a specific directory in Google Drive.
```

In [6]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Ironhack_COLAB/lab-chains-in-langchain/lab-chains-in-langchain-main/Data.csv")


```pseudocode
# Import pandas library.
# Read a CSV file located in Google Drive into a pandas DataFrame.
```

In [7]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


```pseudocode
# Display the first few rows of the DataFrame.
```

## LLMChain

In [10]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community
!pip install langchain==0.2.6 langchain-core==0.2.10 langchain-openai==0.1.7 langchain-community==0.2.6


Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-openai 1.2.1
Uninstalling langchain-openai-1.2.1:
  Successfully uninstalled langchain-openai-1.2.1
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 975.5/975.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.8/332.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 5

```pseudocode
# Uninstall existing LangChain related libraries.
# Install specific versions of LangChain, LangChain Core, LangChain OpenAI, and LangChain Community.
```

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.llm import LLMChain   # 👈 ruta correcta en v1.x


```pseudocode
# Import necessary classes for building LLM chains: ChatOpenAI for the language model, ChatPromptTemplate for defining prompts, and LLMChain for creating a chain.
```

In [11]:
# Cargar la API key desde los secretos de Colab
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Imports correctos con la versión modular
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.llm import LLMChain   # 👈 aquí sí está LLMChain

# Configurar el modelo
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Definir un prompt
prompt = ChatPromptTemplate.from_template("Summarize the review: {review}")

# Crear el chain
chain = LLMChain(llm=llm, prompt=prompt)

# Ejecutar con un ejemplo
result = chain.run({"review": "This mattress had a small hole..."})
print(result)


The review mentions that the mattress had a small hole.


```pseudocode
# Import os and userdata from google.colab.
# Load OpenAI API key from Colab secrets.
# Import ChatOpenAI, ChatPromptTemplate, and LLMChain for updated LangChain.
# Initialize ChatOpenAI model.
# Define a chat prompt template to summarize a review.
# Create an LLMChain with the LLM and prompt.
# Run the chain with a sample review and print the result.
```

In [13]:
#Select a product type to be describe
product = "Mattress"
chain.run(product)

'The reviewer found the mattress to be comfortable and supportive, with good quality materials. They appreciated the cooling properties and the lack of motion transfer. Overall, they were satisfied with their purchase and would recommend the mattress to others.'

```pseudocode
# Define a product string.
# Run the previously created LLMChain with the product as input.
```

## SimpleSequentialChain

In [16]:
from langchain.chains import SimpleSequentialChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.llm import LLMChain

```pseudocode
# Import necessary classes for building SimpleSequentialChain: SimpleSequentialChain, ChatOpenAI, ChatPromptTemplate, and LLMChain.
```

In [17]:
llm = ChatOpenAI(temperature=0.9)

# Prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Write a headline about {topic}"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

# Prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a short poem about the headline: {headline}"
)

# Chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

# SimpleSequentialChain: output of chain_one → input of chain_two
overall_chain = SimpleSequentialChain(chains=[chain_one, chain_two], verbose=True)

# Run the chain
result = overall_chain.run("Artificial Intelligence")
print(result)



> Entering new SimpleSequentialChain chain...
"Revolutionizing Industries: The Impact of Artificial Intelligence on Business"
Artificial intelligence, so advanced and smart,
Revolutionizing industries, playing a key part.
From automation to data analysis,
AI is changing business practices.

No longer are humans alone in decision-making,
AI helps businesses in risk-taking.
Efficiency and productivity on the rise,
Thanks to AI's power and wise.

From healthcare to finance, no industry untouched,
AI's impact on business, truly unmatched.
Embracing this technology, a must for success,
As it revolutionizes industries, nothing less.

> Finished chain.
Artificial intelligence, so advanced and smart,
Revolutionizing industries, playing a key part.
From automation to data analysis,
AI is changing business practices.

No longer are humans alone in decision-making,
AI helps businesses in risk-taking.
Efficiency and productivity on the rise,
Thanks to AI's power and wise.

From healthcare to fin

```pseudocode
# Initialize ChatOpenAI model.
# Define a prompt template for generating a headline about a topic.
# Create the first LLMChain for headlines.
# Define a prompt template for writing a poem based on a headline.
# Create the second LLMChain for poems.
# Construct a SimpleSequentialChain, linking the two chains.
# Run the overall chain with a topic ("Artificial Intelligence") and print the result.
```

In [18]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

```pseudocode
# Create another SimpleSequentialChain using the previously defined chains (chain_one, chain_two).
```

In [19]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
"Sleep Soundly: Discover the Best Mattress For Ultimate Comfort and Support"
In search of peaceful rest we lie,
Seeking a mattress to satisfy,
Ultimate comfort, support divine,
To sleep soundly, a dream so fine.

Layers of softness, firmness too,
A sanctuary for me and you,
Discover the mattress that fits just right,
For a good night's sleep, both day and night.

No more tossing, turning in bed,
With the best mattress, lay your head,
Close your eyes, drift off to sleep,
In a cocoon of comfort, oh so deep.

So rest easy, dear weary soul,
Find the mattress that makes you whole,
Sleep soundly, in comfort and bliss,
For with the best mattress, sweet dreams exist.

> Finished chain.


"In search of peaceful rest we lie,\nSeeking a mattress to satisfy,\nUltimate comfort, support divine,\nTo sleep soundly, a dream so fine.\n\nLayers of softness, firmness too,\nA sanctuary for me and you,\nDiscover the mattress that fits just right,\nFor a good night's sleep, both day and night.\n\nNo more tossing, turning in bed,\nWith the best mattress, lay your head,\nClose your eyes, drift off to sleep,\nIn a cocoon of comfort, oh so deep.\n\nSo rest easy, dear weary soul,\nFind the mattress that makes you whole,\nSleep soundly, in comfort and bliss,\nFor with the best mattress, sweet dreams exist."

```pseudocode
# Run the overall_simple_chain with the 'product' variable as input.
```

**Repeat the above twice for different products**

In [20]:
# Luxury Air Mattress
product1 = "Luxury Air Mattress"
result1 = overall_simple_chain.run(product1)
print(result1)

# Queen Size Sheet Set
product2 = "Queen Size Sheet Set"
result2 = overall_simple_chain.run(product2)
print(result2)




> Entering new SimpleSequentialChain chain...
"Experience the Ultimate Comfort with Luxury Air Mattresses: The Perfect Blend of Support and Style"
Sink into luxurious dreams,
On clouds of comfort, it seems,
With the perfect blend of support,
And style that is a resort.

Experience the ultimate ease,
On these luxury air mattresses please,
Where sleep is a heavenly delight,
And every night is just right.

So lay back, relax, and unwind,
Let your worries be left behind,
For with these mattresses, you'll find,
The perfect comfort for body and mind.

> Finished chain.
Sink into luxurious dreams,
On clouds of comfort, it seems,
With the perfect blend of support,
And style that is a resort.

Experience the ultimate ease,
On these luxury air mattresses please,
Where sleep is a heavenly delight,
And every night is just right.

So lay back, relax, and unwind,
Let your worries be left behind,
For with these mattresses, you'll find,
The perfect comfort for body and mind.


> Entering new SimpleS

```pseudocode
# Define product1 as "Luxury Air Mattress".
# Run the SimpleSequentialChain with product1 and print the result.
# Define product2 as "Queen Size Sheet Set".
# Run the SimpleSequentialChain with product2 and print the result.
```

## SequentialChain

In [30]:
print(df.shape)   # (n_rows, n_cols)


(7, 2)


```pseudocode
# Print the shape (number of rows, number of columns) of the DataFrame.
```

In [31]:
from langchain.chains import SequentialChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.llm import LLMChain

# Initialize LLM
llm = ChatOpenAI(temperature=0.9)

# Prompt 1: translate review
first_prompt = ChatPromptTemplate.from_template("Translate this review into Spanish: {review}")
chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="translated_review")

# Prompt 2: classify sentiment
second_prompt = ChatPromptTemplate.from_template("Classify the sentiment of this text: {translated_review}")
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="sentiment")

# Prompt 3: generate recommendation
third_prompt = ChatPromptTemplate.from_template("Based on the sentiment, give a recommendation: {sentiment}")
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="recommendation")

# Build SequentialChain
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three],
    input_variables=["review"],
    output_variables=["translated_review", "sentiment", "recommendation"],
    verbose=True
)

# Run on two valid reviews (indices 1 and 2)
review1 = df.Review.iloc[1]
result1 = overall_chain.invoke({"review": review1})
print("Review 1 result:", result1)

review2 = df.Review.iloc[2]
result2 = overall_chain.invoke({"review": review2})
print("Review 2 result:", result2)




> Entering new SequentialChain chain...

> Finished chain.
Review 1 result: {'review': 'I loved the waterproof sac, although the opening was made of a hard plastic. I don’t know if that would break easily. But I couldn’t turn my phone on, once it was in the pouch.', 'translated_review': 'Me encantó el saco impermeable, aunque la apertura estaba hecha de plástico duro. No sé si eso se rompería fácilmente. Pero no pude encender mi teléfono una vez que estaba en la bolsa.', 'sentiment': 'Mixed sentiment', 'recommendation': 'It seems like the sentiment is mixed, so it may be best to gather more information or seek a second opinion before making a decision.'}


> Entering new SequentialChain chain...

> Finished chain.
Review 2 result: {'review': "This mattress had a small hole in the top of it (took forever to find where it was), and the patches that they provide did not work, maybe because it's the top of the mattress where it's kind of like fabric and a patch won't stick. Maybe I got u

```pseudocode
# Import necessary classes for building SequentialChain: SequentialChain, ChatOpenAI, ChatPromptTemplate, and LLMChain.
# Initialize ChatOpenAI model.
# Define the first prompt template to translate a review into Spanish.
# Create chain_one for translation.
# Define the second prompt template to classify the sentiment of the translated review.
# Create chain_two for sentiment classification.
# Define the third prompt template to generate a recommendation based on sentiment.
# Create chain_three for recommendations.
# Build a SequentialChain combining chain_one, chain_two, and chain_three, specifying input and output variables.
# Get the review from DataFrame at index 1.
# Run the SequentialChain with review1 and print the results.
# Get the review from DataFrame at index 2.
# Run the SequentialChain with review2 and print the results.
```

**Repeat the above twice for different products or reviews**

In [34]:
# Run SequentialChain on Review at position 1
review1 = df.Review.iloc[1]
result1 = overall_chain.invoke({"review": review1})
print("Result for Review 1:", result1)

# Run SequentialChain on Review at position 2
review2 = df.Review.iloc[2]
result2 = overall_chain.invoke({"review": review2})
print("Result for Review 2:", result2)




> Entering new SequentialChain chain...

> Finished chain.
Result for Review 1: {'review': 'I loved the waterproof sac, although the opening was made of a hard plastic. I don’t know if that would break easily. But I couldn’t turn my phone on, once it was in the pouch.', 'translated_review': 'Me encantó la bolsa impermeable, aunque la apertura estaba hecha de plástico duro. No sé si se rompería fácilmente. Pero no pude encender mi teléfono una vez que estaba en la bolsa.', 'sentiment': 'Neutral', 'recommendation': "It seems like there isn't a strong positive or negative sentiment present. In this case, it would be recommended to gather more information or seek out additional opinions before making a decision."}


> Entering new SequentialChain chain...

> Finished chain.
Result for Review 2: {'review': "This mattress had a small hole in the top of it (took forever to find where it was), and the patches that they provide did not work, maybe because it's the top of the mattress where it

```pseudocode
# Get the review from DataFrame at index 1.
# Run the overall_chain with review1 and print the results.
# Get the review from DataFrame at index 2.
# Run the overall_chain with review2 and print the results.
```

## Router Chain

In [35]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

```pseudocode
# Define a prompt template for physics questions, specifying the AI's persona.
# Define a prompt template for math questions, specifying the AI's persona.
# Define a prompt template for history questions, specifying the AI's persona.
# Define a prompt template for computer science questions, specifying the AI's persona.
# Define a prompt template for biology questions, specifying the AI's persona.
```

In [36]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

```pseudocode
# Create a list of dictionaries, where each dictionary contains the name, description, and prompt template for a specific domain (physics, math, history, computer science, biology).
```

In [37]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

```pseudocode
# Import necessary classes for building a MultiPromptChain: MultiPromptChain, LLMRouterChain, RouterOutputParser from langchain.chains.router, and PromptTemplate.
```

In [38]:
llm = ChatOpenAI(temperature=0)

```pseudocode
# Initialize the ChatOpenAI language model with a temperature of 0 for deterministic output.
```

In [39]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

```pseudocode
# Initialize an empty dictionary for destination chains.
# Iterate through each prompt information dictionary.
# For each, extract the name and prompt template.
# Create a ChatPromptTemplate from the template.
# Create an LLMChain using the LLM and the prompt, and store it in the destination_chains dictionary.
# Create a list of destination descriptions (name: description) for the router.
```

In [40]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

```pseudocode
# Define a default ChatPromptTemplate.
# Create a default LLMChain using the LLM and the default prompt.
```

In [41]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

```pseudocode
# Define a multi-prompt router template string that guides the LLM to select the best prompt for a given input and potentially revise the input.
```

In [42]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

```pseudocode
# Format the Multi-Prompt Router Template with the generated destination descriptions.
# Create a PromptTemplate from the formatted router template, specifying input variables and an output parser.
# Create an LLMRouterChain from the LLM and the router prompt.
```

In [43]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

```pseudocode
# Create a MultiPromptChain, combining the router chain, destination chains, and the default chain, and enable verbose output.
```

In [44]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

```pseudocode
# Run the MultiPromptChain with a physics-related question and print the result.
```

In [45]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

```pseudocode
# Run the MultiPromptChain with a math-related question and print the result.
```

In [46]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells, maintaining the integrity and continuity of the organism's genetic code.\n\nAdditionally, DNA serves as a storage system for genetic information that can be accessed and utilized by cells as needed. This allows for the regulation of gene expression, the repair of damaged DNA, and the adaptation of cells to changing environmental conditions.\n\nIn summary, every cell in our body contains DNA because it is essential fo

```pseudocode
# Run the MultiPromptChain with a biology-related question and print the result.
```

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [47]:
# Math question
print(chain.run("What is the derivative of x^2 + 3x?"))

# History question
print(chain.run("Who was the first emperor of Rome?"))

# Computer science question
print(chain.run("Write a Python function to check if a number is prime."))

# Biology question
print(chain.run("Explain the process of cellular respiration."))




> Entering new MultiPromptChain chain...
math: {'input': 'What is the derivative of x^2 + 3x?'}
> Finished chain.
To find the derivative of x^2 + 3x, we can use the power rule for differentiation. 

The power rule states that if we have a term of the form x^n, the derivative is nx^(n-1). 

So, for the term x^2, the derivative is 2x^(2-1) = 2x. 
And for the term 3x, the derivative is 3x^(1-1) = 3. 

Therefore, the derivative of x^2 + 3x is 2x + 3.


> Entering new MultiPromptChain chain...
History: {'input': 'Who was the first emperor of Rome?'}
> Finished chain.
The first emperor of Rome was Augustus, also known as Caesar Augustus. He ruled from 27 BC until his death in AD 14. Augustus was the adopted son of Julius Caesar and played a crucial role in the transition from the Roman Republic to the Roman Empire.


> Entering new MultiPromptChain chain...
computer science: {'input': 'Write a Python function to check if a number is prime.'}
> Finished chain.
Sure! Here is a Python functio

```pseudocode
# Run the MultiPromptChain with a math question about derivatives and print the result.
# Run the MultiPromptChain with a history question about the first Roman emperor and print the result.
# Run the MultiPromptChain with a computer science question to write a Python prime checking function and print the result.
# Run the MultiPromptChain with a biology question about cellular respiration and print the result.
```

## Summary

This notebook explored various types of LangChain chains:

1.  **LLMChain**: Demonstrated how to create a basic LLMChain to summarize text using a `ChatOpenAI` model and `ChatPromptTemplate`.
2.  **SimpleSequentialChain**: Illustrated how to link two LLMChains where the output of the first becomes the input of the second, creating a sequential flow for tasks like generating a headline and then a poem based on that headline.
3.  **SequentialChain**: Showcased a more advanced sequential chain that allows for explicit input and output variables between multiple linked LLMChains, enabling tasks like translating a review, classifying its sentiment, and generating a recommendation.
4.  **Router Chain (MultiPromptChain)**: Presented how to build a router chain that intelligently directs user queries to the most appropriate specialized LLMChain based on the query's content (e.g., physics, math, history, computer science, biology questions). This involves defining multiple prompt templates, creating chains for each, and using a router to select the correct chain for a given input.

The notebook used `pandas` for data handling and `langchain` components to interact with `ChatOpenAI`.